# Set up a regional MOM6 run within CESM framework (with CRR grid generation)

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [2]:
from pathlib import Path
import sys

In [3]:
%load_ext autoreload
%autoreload 2
sys.path.insert(0, '/glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/crr')
import crocodileregionalruckus as crr
from crocodileregionalruckus.rm6 import regional_mom6 as rmom6

In [6]:
expt = crr.driver.crr_driver.create_experiment_from_config("/glade/u/home/manishrv/documents/nwa12_0.1/configs/nwa_config.json")

2024-10-24 08:41:03,230 - crocodileregionalruckus.driver - INFO - Reading from config file....
2024-10-24 08:41:03,231 - crocodileregionalruckus.driver - INFO - Creating Input and Run Dirs if not already....
2024-10-24 08:41:03,232 - crocodileregionalruckus.driver - INFO - Checking for hgrid and vgrid....
2024-10-24 08:41:03,232 - crocodileregionalruckus.driver - INFO - Found
2024-10-24 08:41:03,233 - crocodileregionalruckus.driver - INFO - hgrid.nc already in dir
2024-10-24 08:41:03,234 - crocodileregionalruckus.driver - INFO - Found
2024-10-24 08:41:03,234 - crocodileregionalruckus.driver - INFO - vcoord.nc already in dir
2024-10-24 08:41:03,235 - crocodileregionalruckus.driver - INFO - Creating Expt Object....
Reading from config file....
Creating Empty Experiment Object....
Setting Default Variables.....
Skipping hgrid and vgrid creation....
Done!
2024-10-24 08:41:03,236 - crocodileregionalruckus.driver - INFO - Checking for bathymetry...
2024-10-24 08:41:03,236 - crocodileregional

In [ ]:
crr.driver.crr_driver.write_config_file(expt, "/glade/u/home/manishrv/documents/nwa12_0.1/nwa_config.json")

# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/abarnes/cesm-runs/visualCaseGen/cesm2_3_beta17_gui/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/rcg-test-1 --machine derecho --run-unsupported --project p93300012 --non-local`

In [5]:
# Save the path where the new CESM config lives
CESM_path = Path(f"/glade/u/home/manishrv/cases/rcg-test-1")

# SECTION 3: Modify the CESM run to make it regional

In [8]:
## Regional CaseGen is part of crr

reg_cas_obj = crr.regional_casegen.cesm_tools.RegionalCaseGen()


reg_cas_obj.setup_cesm(expt, CESM_path, project = "p93300612")
## OR you can explicitly state what variables are needed
# reg_cas_obj.setup_cesm_explicit(hgrid = expt.hgrid, CESMPath = CESM_path,project = "p93300612", mom_input_dir=expt.mom_input_dir, mom_run_dir=expt.mom_run_dir,date_range = expt.date_range)

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 